# GLEE Competition agent V6 — configuration-safe three-family agent

V6 preserves the strongest live-tested policy per family and fixes contract-level and belief-state errors identified from the official `llms.txt` documentation:

- Bargaining keeps V4's 25/25 agreement cycle insurance and V5's Alice calibration, but profiles are now separated by role, information condition, and visible discount configuration so evidence from incompatible games cannot contaminate the acceptance floor.
- Negotiation freezes the V5 policy that raised the displayed rating from 1243.31 to 1344.87 in 55 games with zero fallbacks. Only validation and profile isolation change; the live-tested offer and acceptance constants do not.
- Persuasion retains the successful precision buyer and V5 endgame credibility spending, but corrects a seller-belief error: passed products do not reveal quality to the buyer and therefore no longer update the seller's model of buyer trust. Text recommendations use a conservative negation-aware parser.

The official scoring rule compares payoff percentiles within configuration and role, so V6 optimizes payoff subject to individual rationality and termination rather than agreement alone. No policy can guarantee a higher live rating. Evaluate one family at a time and retain the prior live-tested version if V6 does not improve role-stratified evidence.


In [ ]:
%pip install -q -U glee-sdk



## API key, imports, and thread-safe memory

Named opponents get cross-game profiles. Hidden opponents get game-local memory.
Stable hashing makes mixed strategies reproducible and concurrency-safe.



In [ ]:
import hashlib
import math
import os
import re
import statistics
import threading
from collections import defaultdict, deque
from getpass import getpass

os.environ["GLEE_API_KEY"] = getpass("GLEE API key: ")

LOCK = threading.RLock()
SEEN = set()
DECISION_LOG = deque(maxlen=1500)

BARGAINING_MEMORY = defaultdict(lambda: {
    "rejected_floor": 0.0, "opponent_demands": deque(maxlen=40)
})
NEGOTIATION_MEMORY = defaultdict(lambda: {
    "seller_prices": deque(maxlen=50), "buyer_prices": deque(maxlen=50)
})
# Perspective is part of the key. A named opponent's behavior as seller must not
# be contaminated by observations collected while that opponent was the buyer.
PERSUASION_MEMORY = defaultdict(lambda: {
    "pos_high": 0.0, "pos_low": 0.0,
    "neg_high": 0.0, "neg_low": 0.0,
    "positive_buys": 0.0, "positive_decisions": 0.0,
})


## Shared schema helpers



In [ ]:
def clamp(x, low, high):
    return max(low, min(high, x))

def finite_float(value, default=0.0):
    try:
        number = float(value)
        return number if math.isfinite(number) else default
    except (TypeError, ValueError):
        return default

def round_progress(state):
    current = max(1, int(state.get("round", 1)))
    maximum = state.get("max_rounds")
    if state.get("horizon_known") and maximum:
        return clamp((current - 1) / max(1, int(maximum) - 1), 0.0, 1.0)
    # Unknown horizons concede slowly unless explicit repetition is detected.
    return min(0.65, (current - 1) / 16.0)

def final_round(state):
    return bool(state.get("horizon_known") and state.get("max_rounds") and
                int(state.get("round", 1)) >= int(state["max_rounds"]))

def player_index(player):
    return 1 if player in {"player_1", "alice"} else 2

def canonical_player(player):
    return f"player_{player_index(player)}"

def other_player(player):
    return "player_2" if player_index(player) == 1 else "player_1"

def opponent_key(game, family):
    opponent = game.get("opponent") or {}
    if opponent.get("type") != "hidden" and opponent.get("name"):
        return f"{family}:named:{opponent.get('type')}:{opponent['name']}"
    return f"{family}:game:{game.get('game_id', '')}"

def stable_unit(game, salt=""):
    state = game.get("game_state") or {}
    token = f"{game.get('game_id', '')}:{state.get('round', 1)}:{salt}"
    value = int.from_bytes(hashlib.sha256(token.encode()).digest()[:8], "big")
    return value / 2**64

def logistic(x):
    return 1.0 / (1.0 + math.exp(-clamp(x, -60.0, 60.0)))

def allocation(offer, player):
    keys = (("player_1_gain", "alice_gain") if player_index(player) == 1 else
            ("player_2_gain", "bob_gain"))
    for key in keys:
        if key in offer:
            return finite_float(offer[key], None)
    return None

def action_message(text):
    return str(text)[:2000]

def signal_polarity(value):
    if isinstance(value, dict):
        value = value.get("decision", value.get("message"))
    if value is None:
        return None
    text = str(value).strip().lower().replace("_", " ")
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9' ]+", " ", text)).strip()
    negatives = (
        "do not buy", "don't buy", "should not buy", "shouldn't buy",
        "do not recommend", "don't recommend", "recommend against",
        "not worth", "not a good", "not high quality", "low quality",
        "bad product", "buying would be foolish",
    )
    positives = (
        "buy this", "recommend buying", "recommend this", "worth it",
        "high quality", "great product", "good product", "positive",
    )
    if text in {"no", "false", "negative", "not recommended"}:
        return False
    words = set(text.split())
    if any(phrase in text for phrase in negatives) or words & {"pass", "skip", "avoid"}:
        return False
    if text in {"yes", "true", "recommended"}:
        return True
    if text == "buy" or any(phrase in text for phrase in positives):
        return True
    return None

def tail_repeat_count(values, relative_tolerance=0.002):
    if not values:
        return 0
    last = float(values[-1])
    tolerance = max(1e-9, abs(last) * relative_tolerance)
    count = 0
    for value in reversed(values):
        if abs(float(value) - last) <= tolerance:
            count += 1
        else:
            break
    return count


## 1. Bargaining — role-calibrated surplus capture with cycle insurance

V6 keeps V4's uncapped visible equilibrium and three-cycle escape, which produced 25/25 live agreements, plus V5's Alice-specific payoff shading. Its new safeguard is configuration-scoped learning: a rejection floor is reused only for the same role, information condition, and visible discount signature. This prevents a demanding opponent in one inflation configuration from forcing over-generous offers in a strategically different game. Once repetition is detected, both roles revert to V4's agreement insurance.

For responder share $s$, the proposer maximizes

$$J_i(s)=a(s)(1-s)^{\gamma_i}-(1-a(s))C_{i,t},$$

where $\gamma_A=1.22$, $\gamma_B=1.15$, and $a(s)$ is logistic around the configuration-specific floor. The search floor is $\tau_t-0.018$ for Alice and $\tau_t-0.006$ for Bob.


In [ ]:
def player_delta(state, player, default=0.93):
    return clamp(finite_float(state.get(f"delta_{player_index(player)}", default), default),
                 0.001, 0.9999)

def rubinstein_responder_share(proposer_delta, responder_delta):
    denominator = 1.0 - proposer_delta * responder_delta
    proposer_share = ((1.0 - responder_delta) / denominator
                      if denominator > 1e-12 else 0.5)
    return clamp(1.0 - proposer_share, 0.001, 0.999)

def bargaining_offer_series(state, money):
    series = {"player_1": [], "player_2": []}
    if money <= 0:
        return series
    for record in state.get("history", []):
        if not isinstance(record, dict):
            continue
        offer = record.get("offer") or {}
        proposer = canonical_player(record.get("proposer", offer.get("proposer", "player_1")))
        demand = allocation(offer, proposer)
        if demand is not None:
            series[proposer].append(clamp(demand / money, 0.0, 1.0))
    return series

def bargaining_stall_count(state, money):
    series = bargaining_offer_series(state, money)
    return min(tail_repeat_count(series["player_1"], 0.003),
               tail_repeat_count(series["player_2"], 0.003))

def bargaining_profile_key(game, state, me):
    # Reservation shares are configuration dependent. Never carry a hard
    # rejection floor across roles or incompatible discount conditions.
    info = "complete" if state.get("complete_information") else "hidden"
    own_delta = round(player_delta(state, me), 3)
    opponent = other_player(me)
    visible_opponent_delta = state.get(f"delta_{player_index(opponent)}")
    opponent_delta = (round(finite_float(visible_opponent_delta), 3)
                      if visible_opponent_delta is not None else "x")
    family = (f"bargaining:{canonical_player(me)}:{info}:"
              f"d{own_delta}:od{opponent_delta}")
    return opponent_key(game, family)

def update_bargaining_memory(game, me, opponent, money):
    key = bargaining_profile_key(game, game["game_state"], me)
    with LOCK:
        model = BARGAINING_MEMORY[key]
        for record in game["game_state"].get("history", []):
            if not isinstance(record, dict):
                continue
            offer = record.get("offer") or {}
            proposer = canonical_player(record.get("proposer", offer.get("proposer", "player_1")))
            decision = record.get("decision")
            if isinstance(decision, dict):
                decision = decision.get("decision")
            event = ("bargaining-v6", game.get("game_id"), record.get("round"),
                     proposer, str(decision),
                     tuple(sorted((str(k), str(v)) for k, v in offer.items())))
            if event in SEEN:
                continue
            SEEN.add(event)
            if proposer == canonical_player(me) and str(decision).lower() == "reject":
                rejected = allocation(offer, opponent)
                if rejected is not None and money > 0:
                    model["rejected_floor"] = max(model["rejected_floor"], rejected / money)
            if proposer == canonical_player(opponent):
                demand = allocation(offer, opponent)
                if demand is not None and money > 0:
                    model["opponent_demands"].append(clamp(demand / money, 0.0, 1.0))
        return {"rejected_floor": model["rejected_floor"],
                "opponent_demands": list(model["opponent_demands"])}

def estimate_bargaining_floor(game, state, me, opponent, model):
    t = round_progress(state)
    if state.get("complete_information"):
        prior = rubinstein_responder_share(player_delta(state, me),
                                           player_delta(state, opponent))
    else:
        opponent_type = (game.get("opponent") or {}).get("type")
        prior = (0.46 if opponent_type == "human" else 0.43) + 0.02 * t
    evidence = model["rejected_floor"] + 0.006 if model["rejected_floor"] else 0.0
    if model["opponent_demands"]:
        recent = model["opponent_demands"][-5:]
        demand_floor = statistics.median(recent) - (0.065 - 0.020 * t)
        evidence = max(evidence, demand_floor)
    return clamp(max(prior, evidence), 0.20, 0.999)

def bargaining_strategy(game):
    state = game["game_state"]
    me = canonical_player(game.get("your_player", state["current_player"]))
    opponent = other_player(me)
    money = finite_float(state["money_to_divide"])
    t = round_progress(state)
    my_delta = player_delta(state, me)
    model = update_bargaining_memory(game, me, opponent, money)
    floor = estimate_bargaining_floor(game, state, me, opponent, model)
    stalls = bargaining_stall_count(state, money)

    if game["valid_actions"]["type"] == "offer":
        if stalls >= 3 and model["opponent_demands"]:
            # Match the opponent's revealed demand rather than repeating a split
            # they have already rejected indefinitely.
            responder_share = clamp(model["opponent_demands"][-1], 0.20, 0.9999)
        else:
            best = None
            alice = player_index(me) == 1
            shade = 0.018 if alice else 0.006
            search_floor = clamp(floor - shade, 0.18, 0.999)
            payoff_power = 1.22 if alice else 1.15
            failure_cost = 0.04 + 0.24 * t + 0.55 * (1.0 - my_delta)
            if alice:
                failure_cost *= 0.40
            # 10% through 99.5% in half-percentage-point increments.
            for step in range(20, 200):
                responder_share = step / 200.0
                width = 0.012 if search_floor > 0.80 else 0.022
                probability = logistic((responder_share - search_floor + 0.008) / width)
                own_share = 1.0 - responder_share
                objective = probability * own_share**payoff_power - (1.0 - probability) * failure_cost
                candidate = (objective, own_share, responder_share)
                if best is None or candidate > best:
                    best = candidate
            responder_share = best[2]
        responder_gain = round(money * responder_share, 8)
        own_gain = money - responder_gain
        action = ({"alice_gain": own_gain, "bob_gain": responder_gain}
                  if player_index(me) == 1 else
                  {"alice_gain": responder_gain, "bob_gain": own_gain})
        if state.get("messages_allowed"):
            pct = round(100 * responder_share, 1)
            action["message"] = action_message(
                f"I offer you {pct}% now, accounting for discounting and the observed negotiation path."
            )
        return action

    current_gain = allocation(state.get("last_offer") or {}, me)
    if current_gain is None:
        return {"decision": "reject"}
    if final_round(state):
        return {"decision": "accept" if current_gain >= 0 else "reject"}
    if stalls >= 3 and current_gain > 0:
        return {"decision": "accept"}
    next_own_share = 1.0 - floor
    deal_probability = clamp(0.80 + 0.10 * t - 0.20 * model["rejected_floor"], 0.45, 0.92)
    continuation_share = my_delta * next_own_share * deal_probability
    role_margin = 0.025 * (1.0 - t) if player_index(me) == 1 and stalls == 0 else 0.0
    risk_floor = max(0.0, 0.34 + role_margin - 0.08 * t - 0.07 * stalls)
    required = money * max(risk_floor, continuation_share)
    return {"decision": "accept" if current_gain + 1e-9 >= required else "reject"}


## 2. Negotiation — role-calibrated concession forecasting

V6 freezes V5's live-tested negotiation economics: the 55-game V5 batch gained 101.56 displayed rating points with zero fallbacks. Complete-information sellers use the 74%-to-60% capture schedule; buyers begin at 66% and concede toward 56%. Under hidden values, sellers retain the V4 target while buyers open closer to their value and discount continuation slightly more. V6 changes only safety plumbing and profile namespaces here, minimizing regression risk.

$$c_t^{seller}=0.74-0.14t,\qquad c_t^{buyer}=0.66-0.10t.$$

For target utility $U_t$ and stall count $k$, the acceptance continuation value is

$$U_{cont}=U_t\left(b_i-0.12t-0.08\min(k,2)\right),\quad b_s=0.80,\ b_b=0.75.$$


In [ ]:
def offer_sender(item, record, field):
    sender = item.get("from_player") if isinstance(item, dict) else None
    if sender:
        return canonical_player(sender)
    if field == "counteroffer" and record.get("decided_by"):
        return canonical_player(record["decided_by"])
    return None

def negotiation_price_series(state):
    series = {"player_1": [], "player_2": []}
    for record in state.get("history", []):
        if not isinstance(record, dict):
            continue
        for field in ("offer", "counteroffer"):
            item = record.get(field)
            if isinstance(item, dict) and item.get("price") is not None:
                sender = offer_sender(item, record, field)
                if sender:
                    series[sender].append(finite_float(item["price"]))
    return series

def negotiation_stall_count(state, me, opponent):
    series = negotiation_price_series(state)
    return min(tail_repeat_count(series[canonical_player(me)], 0.001),
               tail_repeat_count(series[canonical_player(opponent)], 0.001))

def negotiation_profile_key(game, state):
    me = canonical_player(game.get("your_player", state["current_player"]))
    role = state.get(f"{me}_role", "unknown")
    info = "complete" if state.get("complete_information") else "hidden"
    return opponent_key(game, f"negotiation:{role}:{info}")

def update_negotiation_memory(game, state):
    key = negotiation_profile_key(game, state)
    with LOCK:
        model = NEGOTIATION_MEMORY[key]
        for record in state.get("history", []):
            if not isinstance(record, dict):
                continue
            for field in ("offer", "counteroffer"):
                item = record.get(field)
                if not isinstance(item, dict) or item.get("price") is None:
                    continue
                sender = offer_sender(item, record, field)
                if sender is None:
                    continue
                price = finite_float(item["price"])
                event = ("negotiation-v6", game.get("game_id"), record.get("round"),
                         field, sender, price)
                if event in SEEN:
                    continue
                SEEN.add(event)
                role = state.get(f"{sender}_role")
                if role in {"seller", "buyer"}:
                    model[f"{role}_prices"].append(price)
        return {name: list(values) for name, values in model.items()}

def opponent_prices_in_game(state, opponent):
    return negotiation_price_series(state)[canonical_player(opponent)]

def projected_opponent_price(prices, role):
    if not prices:
        return None
    latest = prices[-1]
    if len(prices) < 2:
        return latest
    step = latest - prices[-2]
    # Only extrapolate concessions in the economically expected direction.
    if role == "seller":
        step = min(0.0, step)
    else:
        step = max(0.0, step)
    return max(0.0, latest + 0.6 * step)

def negotiation_strategy(game):
    state = game["game_state"]
    me = canonical_player(game.get("your_player", state["current_player"]))
    opponent = other_player(me)
    role = state[f"{me}_role"]
    opponent_role = state[f"{opponent}_role"]
    my_value = finite_float(state[f"{me}_value"])
    t = round_progress(state)
    update_negotiation_memory(game, state)
    observed = opponent_prices_in_game(state, opponent)
    stalls = negotiation_stall_count(state, me, opponent)
    opponent_value = state.get(f"{opponent}_value")
    surplus = None

    if state.get("complete_information") and opponent_value is not None:
        opponent_value = finite_float(opponent_value)
        seller_value = my_value if role == "seller" else opponent_value
        buyer_value = my_value if role == "buyer" else opponent_value
        surplus = buyer_value - seller_value
        own_capture = ((0.74 - 0.14 * t) if role == "seller" else
                       (0.66 - 0.10 * t))
        if observed and surplus > 1e-12:
            last_price = observed[-1]
            opponent_demand = ((last_price - seller_value) / surplus if opponent_role == "seller"
                               else (buyer_value - last_price) / surplus)
            feasible_capture = 1.0 - clamp(opponent_demand - (0.05 + 0.04 * t), 0.0, 1.0)
            own_capture = 0.58 * own_capture + 0.42 * feasible_capture
        own_capture = clamp(own_capture, 0.52, 0.82)
        if surplus <= 0:
            target = my_value
        elif role == "seller":
            target = seller_value + own_capture * surplus
        else:
            target = buyer_value - own_capture * surplus
    elif observed:
        anchor = projected_opponent_price(observed, opponent_role)
        claim = ((0.70 - 0.12 * t) if role == "seller" else
                 (0.62 - 0.10 * t))
        if role == "seller" and anchor >= my_value:
            target = my_value + claim * (anchor - my_value)
        elif role == "buyer" and anchor <= my_value:
            target = my_value - claim * (my_value - anchor)
        else:
            target = my_value
    elif role == "seller":
        target = my_value * (1.36 - 0.16 * t)
    else:
        target = my_value * (0.80 + 0.10 * t)

    target = max(0.0, finite_float(target, my_value))
    if game["valid_actions"]["type"] == "offer":
        action = {"product_price": round(target, 8)}
        if state.get("messages_allowed"):
            action["message"] = action_message(
                "This price is inside the feasible interval revealed by our offers."
            )
        return action

    price = finite_float((state.get("last_offer") or {}).get("price"), my_value)
    offered_utility = price - my_value if role == "seller" else my_value - price
    profitable = offered_utility >= -1e-9
    if final_round(state):
        return {"decision": "AcceptOffer" if profitable else "RejectOffer"}
    if profitable and stalls >= 2:
        return {"decision": "AcceptOffer"}
    if not profitable and stalls >= 3 and not state.get("horizon_known"):
        return {"decision": "WalkAway"}

    target_utility = abs(target - my_value)
    offered_capture = offered_utility / surplus if surplus is not None and surplus > 1e-12 else None
    continuation_base = 0.80 if role == "seller" else 0.75
    continuation = target_utility * (continuation_base - 0.12 * t -
                                     0.08 * min(stalls, 2))
    if profitable and (offered_utility + 1e-9 >= continuation or
                       (offered_capture is not None and offered_capture >= 0.50 + 0.05 * (1 - t))):
        return {"decision": "AcceptOffer"}

    blend = 0.24 + 0.43 * t + 0.08 * min(stalls, 2)
    blend = clamp(blend, 0.0, 0.78)
    counter = (1.0 - blend) * target + blend * price
    counter = max(my_value, counter) if role == "seller" else min(my_value, counter)
    action = {"decision": "RejectOffer", "product_price": round(max(0.0, counter), 8)}
    if state.get("messages_allowed"):
        action["message"] = action_message(
            "I am conceding inside my individually rational range."
        )
    return action


## 3. Persuasion — observation-correct trust and finite credibility allocation

V6 preserves V3/V4's direct estimate of `P(high | signal)` for the successful buyer role, caps persistent evidence, and gives current-game revealed outcomes extra weight. Its text parser checks negation before positive language and treats ambiguous rhetoric as uninformative rather than trusting any sentence containing the word `buy`.

For the seller, V6 follows the official observation rule exactly: only a purchased product reveals quality to the buyer, so passed products never update modeled buyer trust. If the buyer's posterior cutoff is $c$, define

$$A=4\rho_0+n_H,\qquad N=4+n_H+n_L,\qquad B=\max\left(0,\frac{A}{c}-N\right).$$

$B$ is the remaining low-quality credibility budget. V6 schedules it over the current and expected future low-quality opportunities, becoming progressively more willing to spend reputation near the final round.


In [ ]:
def update_persuasion_memory(game, seller_view):
    state = game["game_state"]
    perspective = "buyer-response" if seller_view else "seller-reliability"
    action_type = game.get("valid_actions", {}).get("type", "")
    mode = state.get("seller_message_type") or (
        "text" if action_type == "seller_message" else "binary"
    )
    key = opponent_key(game, f"persuasion:{perspective}:{mode}")
    with LOCK:
        model = PERSUASION_MEMORY[key]
        for record in state.get("history", []):
            if not isinstance(record, dict):
                continue
            signal = signal_polarity(record.get("seller_message"))
            quality = record.get("quality")
            decision = record.get("buyer_decision")
            if isinstance(decision, dict):
                decision = decision.get("decision")
            event = ("persuasion-v6", perspective, game.get("game_id"),
                     record.get("round"), signal, quality, str(decision))
            if event in SEEN:
                continue
            SEEN.add(event)
            bought = (record.get("bought") is True or
                      str(decision).lower() == "yes")
            # A seller sees every quality, but the buyer learns quality only
            # after buying. Seller-side trust must use the buyer's information.
            quality_visible_to_buyer = (not seller_view) or bought
            if (signal is not None and quality in {"high", "low"} and
                    quality_visible_to_buyer):
                model[f"{'pos' if signal else 'neg'}_{quality}"] += 1.0
            if seller_view and signal is True and str(decision).lower() in {"yes", "no"}:
                model["positive_decisions"] += 1.0
                model["positive_buys"] += float(str(decision).lower() == "yes")
        return dict(model)

def current_signal_counts(history, positive):
    high = low = 0.0
    for record in history or []:
        if not isinstance(record, dict):
            continue
        signal = signal_polarity(record.get("seller_message"))
        quality = record.get("quality")
        if signal is positive and quality == "high":
            high += 1.0
        elif signal is positive and quality == "low":
            low += 1.0
    return high, low

def strategic_signal_prior(positive, p):
    q_high, q_low = ((0.90, 0.24) if positive else (0.10, 0.76))
    denominator = p * q_high + (1.0 - p) * q_low
    return p * q_high / denominator if denominator > 1e-12 else p

def smoothed_signal_precision(model, positive, p, history=None):
    if positive:
        high, low = model["pos_high"], model["pos_low"]
    else:
        high, low = model["neg_high"], model["neg_low"]
    # Cap persistent effective sample size so named-opponent reputation can adapt.
    total = high + low
    if total > 12.0:
        scale = 12.0 / total
        high, low = high * scale, low * scale
    recent_high, recent_low = current_signal_counts(history, positive)
    prior = strategic_signal_prior(positive, p)
    strength = 4.0
    return ((strength * prior + high + 1.5 * recent_high) /
            (strength + high + low + 1.5 * (recent_high + recent_low)))

def positive_trust(model, p):
    prior = strategic_signal_prior(True, p)
    return ((4.0 * prior + model["pos_high"]) /
            (4.0 + model["pos_high"] + model["pos_low"]))

def inferred_hidden_cutoff(history, p):
    prior = strategic_signal_prior(True, p)
    high = low = 0.0
    buy_levels, pass_levels = [], []
    for record in history or []:
        if not isinstance(record, dict):
            continue
        signal = signal_polarity(record.get("seller_message"))
        decision = record.get("buyer_decision")
        if isinstance(decision, dict):
            decision = decision.get("decision")
        trust = (4.0 * prior + high) / (4.0 + high + low)
        if signal is True and str(decision).lower() == "yes":
            buy_levels.append(trust)
        elif signal is True and str(decision).lower() == "no":
            pass_levels.append(trust)
        quality = record.get("quality")
        bought = (record.get("bought") is True or
                  str(decision).lower() == "yes")
        if bought and signal is True and quality == "high":
            high += 1.0
        elif bought and signal is True and quality == "low":
            low += 1.0
    if buy_levels:
        estimate = min(buy_levels)
        if pass_levels:
            estimate = max(estimate, 0.5 * (max(pass_levels) + estimate))
        return clamp(estimate, 0.50, 0.95)
    return None

def credibility_budget_probability(model, p, cutoff, remaining, t,
                                   response_rate, cap):
    prior = strategic_signal_prior(True, p)
    alpha = 4.0 * prior + model["pos_high"]
    total = 4.0 + model["pos_high"] + model["pos_low"]
    # Maximum additional revealed low qualities before posterior precision
    # crosses the buyer's cutoff. Spread them over expected remaining lows.
    credibility = max(0.0, alpha / max(cutoff, 1e-6) - total)
    opportunities = 1.0 + max(0, remaining) * (1.0 - p)
    scheduled = credibility / max(1.0, opportunities)
    urgency = 0.78 + 0.32 * t
    responsiveness = 0.72 + 0.38 * response_rate
    return clamp(scheduled * urgency * responsiveness, 0.0, cap)

def seller_low_pool_probability(game, state, model, p, price, t, round_no, total_rounds):
    response_rate = ((model["positive_buys"] + 1.5) /
                     (model["positive_decisions"] + 2.0))
    if model["positive_decisions"] >= 3 and response_rate < 0.28:
        return 0.0
    if round_no <= max(1, math.ceil(0.10 * total_rounds)):
        return 0.0

    trust = positive_trust(model, p)
    remaining = total_rounds - round_no
    if "v" in state and "u" in state:
        v, u = finite_float(state["v"]), finite_float(state["u"])
        if price <= u:
            return 1.0
        if price >= v or not (0.0 < p < 1.0) or v <= u:
            return 0.0
        cutoff = clamp((price - u) / (v - u), 1e-6, 1.0 - 1e-6)
        if remaining == 0 and trust >= cutoff - 0.03 and response_rate >= 0.35:
            return 1.0
        margin = 0.06 * (1.0 - t)
        if trust < cutoff + margin:
            return 0.0
        return credibility_budget_probability(
            model, p, cutoff, remaining, t, response_rate, 0.92
        )

    cutoff = inferred_hidden_cutoff(state.get("history", []), p)
    if cutoff is None or model["pos_high"] < 1 or response_rate < 0.40:
        return 0.0
    if remaining == 0 and trust >= cutoff - 0.04:
        return 1.0
    margin = 0.07 * (1.0 - t)
    if trust < cutoff + margin:
        return 0.0
    return credibility_budget_probability(
        model, p, cutoff, remaining, t, response_rate, 0.72
    )

def persuasion_strategy(game):
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    price = finite_float(state["product_price"])
    p = clamp(finite_float(state.get("p"), 0.5), 0.0, 1.0)
    round_no = max(1, int(state.get("round", 1)))
    total_rounds = max(round_no, int(state.get("total_rounds", round_no)))
    t = (round_no - 1) / max(1, total_rounds - 1)
    seller_view = action_type in {"seller_message", "seller_recommendation"}
    model = update_persuasion_memory(game, seller_view)

    if seller_view:
        quality = state.get("current_quality")
        recommend = quality == "high"
        if quality == "low":
            probability = seller_low_pool_probability(
                game, state, model, p, price, t, round_no, total_rounds
            )
            recommend = stable_unit(game, "v6-seller-pool") < probability
        if action_type == "seller_recommendation":
            return {"decision": "yes" if recommend else "no"}
        return {"message": action_message(
            "My recommendation: BUY this product." if recommend else
            "My recommendation: PASS on this product this round."
        )}

    v, u = finite_float(state["v"]), finite_float(state["u"])
    if price <= u:
        return {"decision": "yes"}
    if price > v:
        return {"decision": "no"}
    signal = signal_polarity(state.get("seller_message"))
    posterior = (p if signal is None else
                 smoothed_signal_precision(model, signal, p, state.get("history", [])))
    expected_value = posterior * v + (1.0 - posterior) * u
    observations = (model["pos_high"] + model["pos_low"] if signal is True else
                    model["neg_high"] + model["neg_low"] if signal is False else 0.0)
    remaining_fraction = (total_rounds - round_no) / max(1, total_rounds)
    information_bonus = (0.010 * max(0.0, v - u) * remaining_fraction /
                         math.sqrt(1.0 + observations) if signal is True else 0.0)
    return {"decision": "yes" if expected_value + information_bonus >= price else "no"}


## Validated dispatcher and safe fallback

Invalid moves and turn timeouts are scored at the fifth percentile. Every V3 move is
therefore validated locally; unexpected schemas fall back to a conservative legal
action and are recorded in `DECISION_LOG`.



In [ ]:
STRATEGIES = {
    "bargaining": bargaining_strategy,
    "negotiation": negotiation_strategy,
    "persuasion": persuasion_strategy,
}

def fallback_action(game):
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    if family == "bargaining":
        if action_type == "offer":
            money = finite_float(state["money_to_divide"])
            alice = round(money / 2.0, 8)
            return {"alice_gain": alice, "bob_gain": money - alice}
        return {"decision": "accept"}
    if family == "negotiation":
        me = canonical_player(game.get("your_player", state["current_player"]))
        role = state[f"{me}_role"]
        value = finite_float(state[f"{me}_value"])
        if action_type == "offer":
            return {"product_price": max(0.0, value)}
        price = finite_float((state.get("last_offer") or {}).get("price"), value)
        profitable = price >= value if role == "seller" else price <= value
        if profitable:
            return {"decision": "AcceptOffer"}
        if final_round(state):
            return {"decision": "RejectOffer"}
        return {"decision": "RejectOffer", "product_price": max(0.0, value)}
    if action_type == "seller_message":
        return {"message": "My recommendation: PASS this round."}
    if action_type == "seller_recommendation":
        return {"decision": "no"}
    p = finite_float(state.get("p"), 0.5)
    expected = p * finite_float(state.get("v")) + (1 - p) * finite_float(state.get("u"))
    return {"decision": "yes" if expected >= finite_float(state["product_price"]) else "no"}

def is_finite_number(value):
    return (isinstance(value, (int, float)) and not isinstance(value, bool) and
            math.isfinite(float(value)))

def contract_action_keys(game, action):
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    if family == "bargaining":
        keys = {"alice_gain", "bob_gain"} if action_type == "offer" else {"decision"}
    elif family == "negotiation":
        if action_type == "offer":
            keys = {"product_price"}
        else:
            keys = {"decision"}
            if action.get("decision") == "RejectOffer" and not final_round(state):
                keys.add("product_price")
    elif action_type == "seller_message":
        keys = {"message"}
    else:
        keys = {"decision"}
    if (family in {"bargaining", "negotiation"} and
            state.get("messages_allowed")):
        keys.add("message")
    return keys

def validate_action(game, action):
    if not isinstance(action, dict):
        raise ValueError("strategy must return a dict")
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    allowed = contract_action_keys(game, action)
    if set(action) - allowed:
        raise ValueError(f"unexpected action keys: {sorted(set(action) - allowed)}")
    declared = game["valid_actions"].get("fields")
    if isinstance(declared, dict) and declared:
        undeclared = set(action) - set(declared)
        if undeclared:
            raise ValueError(f"keys absent from valid_actions.fields: {sorted(undeclared)}")
    if "message" in action and not isinstance(action["message"], str):
        raise ValueError("message must be a string")
    if family == "bargaining" and action_type == "offer":
        if not is_finite_number(action.get("alice_gain")) or not is_finite_number(action.get("bob_gain")):
            raise ValueError("bargaining gains must be finite numbers")
        alice = float(action["alice_gain"])
        bob = float(action["bob_gain"])
        pot = finite_float(state["money_to_divide"])
        if not all(math.isfinite(x) and x >= 0 for x in (alice, bob)):
            raise ValueError("invalid bargaining allocation")
        if not math.isclose(alice + bob, pot, rel_tol=1e-10, abs_tol=1e-7):
            raise ValueError("bargaining gains do not sum to the pot")
    elif family == "bargaining":
        if action.get("decision") not in {"accept", "reject", "walkaway"}:
            raise ValueError("invalid bargaining decision")
    elif family == "negotiation" and action_type == "offer":
        if (not is_finite_number(action.get("product_price")) or
                float(action["product_price"]) < 0):
            raise ValueError("invalid negotiation price")
    elif family == "negotiation":
        if action.get("decision") not in {"AcceptOffer", "RejectOffer", "WalkAway"}:
            raise ValueError("invalid negotiation decision")
        if action["decision"] == "RejectOffer" and not final_round(state):
            if (not is_finite_number(action.get("product_price")) or
                    float(action["product_price"]) < 0):
                raise ValueError("counteroffer required")
    elif action_type == "seller_message":
        if not isinstance(action.get("message"), str) or len(action["message"]) > 2000:
            raise ValueError("invalid persuasion message")
    elif action.get("decision") not in {"yes", "no"}:
        raise ValueError("invalid persuasion decision")
    if "message" in action and len(action["message"]) > 2000:
        raise ValueError("message exceeds 2,000 characters")
    return action

def strategy(game):
    error = None
    try:
        family = game["game_family"]
        action = validate_action(game, STRATEGIES[family](game))
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        action = validate_action(game, fallback_action(game))
        print(f"SAFE FALLBACK {game.get('game_id')}: {error}")
    with LOCK:
        DECISION_LOG.append({
            "game_id": game.get("game_id"), "family": game.get("game_family"),
            "round": (game.get("game_state") or {}).get("round"),
            "action": dict(action), "error": error,
        })
    return action



## Offline schema, history-regression, and role-calibration tests

The V4 and V5 regression suites remain intact. V6 adds official-contract tests for purchased-only quality revelation, negation-aware text parsing, configuration-scoped bargaining memory, finite credibility allocation, strict numeric action types, exact allocations across pot scales, and feasible negotiation offers across positive and negative surplus states.


In [ ]:
def base_game(family, action_type, state, player="player_1", game_id="test",
              opponent=None):
    return {
        "game_id": game_id, "game_family": family, "your_player": player,
        "opponent": opponent or {"type": "hidden", "name": None},
        "valid_actions": {"type": action_type, "fields": {}},
        "game_state": state,
    }

def repeated_bargaining_history(cycles=4):
    history = []
    for index in range(cycles):
        history.append({"round": 2 * index + 1, "proposer": "player_1",
                        "offer": {"player_1_gain": 35, "player_2_gain": 65},
                        "decision": "reject"})
        history.append({"round": 2 * index + 2, "proposer": "player_2",
                        "offer": {"player_1_gain": 0.01, "player_2_gain": 99.99},
                        "decision": "reject"})
    return history

def repeated_negotiation_history(price_seller, price_buyer, cycles=4):
    return [{"round": i + 1,
             "offer": {"price": price_seller, "from_player": "player_1"},
             "decision": "RejectOffer", "decided_by": "player_2",
             "counteroffer": {"price": price_buyer, "from_player": "player_2"}}
            for i in range(cycles)]

def run_smoke_tests():
    for player in ("player_1", "player_2"):
        state = {"current_player": player, "round": 1, "max_rounds": 5,
                 "horizon_known": True, "money_to_divide": 100,
                 "delta_1": 0.9, "delta_2": 0.95,
                 "complete_information": True, "history": [],
                 "messages_allowed": True}
        action = strategy(base_game("bargaining", "offer", state, player,
                                    f"test-b-{player}"))
        assert math.isclose(action["alice_gain"] + action["bob_gain"], 100)

    # Regression for the observed 10% versus 0% inflation failure. The visible
    # equilibrium requires giving the almost-undiscounted responder over 90%.
    extreme = {"current_player": "player_1", "round": 1,
               "horizon_known": False, "money_to_divide": 100,
               "delta_1": 0.9, "delta_2": 0.999,
               "complete_information": True, "history": []}
    extreme_action = strategy(base_game("bargaining", "offer", extreme,
                                        "player_1", "test-b-extreme"))
    assert extreme_action["bob_gain"] > 90

    cycle_state = dict(extreme, current_player="player_1", round=9,
                       history=repeated_bargaining_history(),
                       last_offer={"player_1_gain": 0.01, "player_2_gain": 99.99})
    assert strategy(base_game("bargaining", "decision", cycle_state,
                              "player_1", "test-b-cycle"))["decision"] == "accept"

    full_negotiation = {"current_player": "player_1",
        "player_1_role": "seller", "player_2_role": "buyer",
        "player_1_value": 40, "player_2_value": 100,
        "complete_information": True, "round": 1, "max_rounds": 5,
        "horizon_known": True, "history": [], "messages_allowed": True}
    price = strategy(base_game("negotiation", "offer", full_negotiation,
                               "player_1", "test-n-full"))["product_price"]
    assert 40 <= price <= 100

    # Repeated 101/100 prices are infeasible for a buyer valued at 100. V4 exits
    # safely rather than polling through 99 rounds or accepting negative utility.
    stalled = {"current_player": "player_2", "player_1_role": "seller",
        "player_2_role": "buyer", "player_2_value": 100,
        "complete_information": False, "round": 9, "horizon_known": False,
        "last_offer": {"price": 101, "from_player": "player_1"},
        "history": repeated_negotiation_history(101, 100)}
    assert strategy(base_game("negotiation", "decision", stalled,
                              "player_2", "test-n-stall"))["decision"] == "WalkAway"
    stalled_at_value = dict(stalled, last_offer={"price": 100, "from_player": "player_1"},
                            history=repeated_negotiation_history(100, 100))
    assert strategy(base_game("negotiation", "decision", stalled_at_value,
                              "player_2", "test-n-zero"))["decision"] == "AcceptOffer"

    high_product = {"current_quality": "high", "product_price": 50, "p": 0.5,
                    "v": 100, "u": 0, "round": 1, "total_rounds": 10,
                    "history": []}
    assert strategy(base_game("persuasion", "seller_recommendation", high_product,
                              "player_1", "test-p-high")) == {"decision": "yes"}
    expensive = {"seller_message": "I recommend buying this product.",
                 "product_price": 101, "p": 0.9, "v": 100, "u": 0,
                 "round": 1, "total_rounds": 5, "history": []}
    assert strategy(base_game("persuasion", "buyer_decision", expensive,
                              "player_2", "test-p-expensive")) == {"decision": "no"}

    # The same named opponent receives separate memory namespaces by role.
    named = {"type": "agent", "name": "same-opponent"}
    seller_history = dict(high_product, current_quality="low", round=3,
        history=[{"round": 1, "seller_message": {"decision": "yes"},
                  "buyer_decision": "yes", "quality": "high"}])
    strategy(base_game("persuasion", "seller_recommendation", seller_history,
                       "player_1", "test-p-role-a", named))
    buyer_history = {"seller_message": {"decision": "yes"}, "product_price": 40,
        "p": 0.5, "v": 100, "u": 0, "round": 2, "total_rounds": 5,
        "history": [{"round": 1, "seller_message": {"decision": "yes"},
                     "buyer_decision": "yes", "quality": "low"}]}
    strategy(base_game("persuasion", "buyer_decision", buyer_history,
                       "player_2", "test-p-role-b", named))
    buyer_key = opponent_key(base_game("persuasion", "seller_recommendation",
                              seller_history, "player_1", "x", named),
                             "persuasion:buyer-response:binary")
    reliability_key = opponent_key(base_game("persuasion", "buyer_decision",
                                    buyer_history, "player_2", "y", named),
                                   "persuasion:seller-reliability:binary")
    assert buyer_key != reliability_key
    assert PERSUASION_MEMORY[buyer_key]["pos_high"] == 1
    assert PERSUASION_MEMORY[reliability_key]["pos_low"] == 1

    errors = [entry for entry in DECISION_LOG if entry["error"]]
    assert not errors, errors
    print("All V4 schema and history-regression tests passed.")

run_smoke_tests()


def run_v5_calibration_tests():
    extreme = {"current_player": "player_1", "round": 1,
               "horizon_known": False, "money_to_divide": 100,
               "delta_1": 0.9, "delta_2": 0.999,
               "complete_information": True, "history": []}
    action = strategy(base_game("bargaining", "offer", extreme,
                                "player_1", "v5-alice-extreme"))
    assert 95 <= action["bob_gain"] < 99.5

    buyer_state = {"current_player": "player_2", "player_1_role": "seller",
        "player_2_role": "buyer", "player_1_value": 40, "player_2_value": 100,
        "complete_information": True, "round": 1, "max_rounds": 5,
        "horizon_known": True, "history": []}
    price = strategy(base_game("negotiation", "offer", buyer_state,
                               "player_2", "v5-buyer-full"))["product_price"]
    assert 40 <= price <= 100

    trusted_history = [
        {"round": i + 1, "seller_message": {"decision": "yes"},
         "buyer_decision": "yes", "quality": "high"}
        for i in range(5)
    ]
    final_low = {"current_quality": "low", "product_price": 50, "p": 0.5,
                 "v": 100, "u": 0, "round": 10, "total_rounds": 10,
                 "history": trusted_history}
    assert strategy(base_game("persuasion", "seller_recommendation", final_low,
                              "player_1", "v5-final-pool")) == {"decision": "yes"}

    recent_lies = [
        {"round": i + 1, "seller_message": {"decision": "yes"},
         "buyer_decision": "yes", "quality": "low"}
        for i in range(3)
    ]
    skeptical = {"seller_message": {"decision": "yes"}, "product_price": 70,
                 "p": 0.5, "v": 100, "u": 0, "round": 4,
                 "total_rounds": 8, "history": recent_lies}
    assert strategy(base_game("persuasion", "buyer_decision", skeptical,
                              "player_2", "v5-recent-lies")) == {"decision": "no"}
    errors = [entry for entry in DECISION_LOG if entry["error"]]
    assert not errors, errors
    print("All V5 role-calibration tests passed.")

run_v5_calibration_tests()


def run_v6_contract_tests():
    # Official observation rule: a passed product does not reveal quality.
    named_pass = {"type": "agent", "name": "v6-pass-audit"}
    passed = {"current_quality": "high", "product_price": 50,
              "p": 0.5, "v": 100, "u": 0, "round": 2,
              "total_rounds": 5, "history": [
                  {"round": 1, "seller_message": {"decision": "yes"},
                   "buyer_decision": "no", "bought": False,
                   "quality": "low"}]}
    pass_game = base_game("persuasion", "seller_recommendation", passed,
                          "player_1", "v6-pass", named_pass)
    strategy(pass_game)
    pass_key = opponent_key(pass_game, "persuasion:buyer-response:binary")
    assert PERSUASION_MEMORY[pass_key]["pos_low"] == 0
    assert PERSUASION_MEMORY[pass_key]["positive_decisions"] == 1

    named_buy = {"type": "agent", "name": "v6-buy-audit"}
    bought = dict(passed, history=[
        {"round": 1, "seller_message": {"decision": "yes"},
         "buyer_decision": "yes", "bought": True, "quality": "low"}
    ])
    buy_game = base_game("persuasion", "seller_recommendation", bought,
                         "player_1", "v6-buy", named_buy)
    strategy(buy_game)
    buy_key = opponent_key(buy_game, "persuasion:buyer-response:binary")
    assert PERSUASION_MEMORY[buy_key]["pos_low"] == 1

    assert signal_polarity("Buying would be foolish; do not buy.") is False
    assert signal_polarity("My recommendation: BUY this product.") is True
    assert signal_polarity("Consider the available information carefully.") is None
    assert signal_polarity("A compassionate description.") is None

    model = {"pos_high": 1.0, "pos_low": 0.0}
    early = credibility_budget_probability(model, 0.5, 0.7, 9, 0.1, 0.8, 0.92)
    late = credibility_budget_probability(model, 0.5, 0.7, 1, 0.9, 0.8, 0.92)
    assert 0 <= early < late <= 0.92

    # Rejection floors must not leak across roles or discount configurations.
    template = {"current_player": "player_1", "round": 1,
                "horizon_known": True, "max_rounds": 5,
                "money_to_divide": 100, "delta_1": 0.9,
                "delta_2": 0.95, "complete_information": True,
                "history": []}
    key_a = bargaining_profile_key(base_game("bargaining", "offer", template,
                                              "player_1", "cfg-a", named_buy),
                                    template, "player_1")
    changed = dict(template, delta_1=0.8)
    key_b = bargaining_profile_key(base_game("bargaining", "offer", changed,
                                              "player_1", "cfg-b", named_buy),
                                    changed, "player_1")
    assert key_a != key_b

    # Property sweeps: exact bargaining budgets and feasible negotiation.
    for pot in (1.0, 37.5, 100.0, 1_000_000.0):
        for d1, d2 in ((0.5, 0.5), (0.9, 0.999), (0.999, 0.7)):
            for player in ("player_1", "player_2"):
                state = dict(template, current_player=player, money_to_divide=pot,
                             delta_1=d1, delta_2=d2)
                action = strategy(base_game("bargaining", "offer", state, player,
                                            f"v6-sweep-b-{pot}-{d1}-{d2}-{player}"))
                assert action["alice_gain"] >= 0 and action["bob_gain"] >= 0
                assert math.isclose(action["alice_gain"] + action["bob_gain"],
                                    pot, rel_tol=1e-10, abs_tol=1e-7)

    for seller_value, buyer_value in ((0, 100), (40, 100), (100, 40), (99, 100)):
        for player in ("player_1", "player_2"):
            state = {"current_player": player, "player_1_role": "seller",
                     "player_2_role": "buyer", "player_1_value": seller_value,
                     "player_2_value": buyer_value, "complete_information": True,
                     "round": 1, "max_rounds": 5, "horizon_known": True,
                     "history": [], "messages_allowed": False}
            action = strategy(base_game("negotiation", "offer", state, player,
                                        f"v6-sweep-n-{seller_value}-{buyer_value}-{player}"))
            assert action["product_price"] >= 0
            if buyer_value >= seller_value:
                assert seller_value <= action["product_price"] <= buyer_value

    # Strict local validation catches server-invalid types before submission.
    invalid = base_game("bargaining", "offer", template, "player_1", "bad-bool")
    try:
        validate_action(invalid, {"alice_gain": True, "bob_gain": 99})
        raise AssertionError("boolean gain was accepted")
    except ValueError:
        pass

    errors = [entry for entry in DECISION_LOG if entry["error"]]
    assert not errors, errors
    print("All V6 contract, observation, and property tests passed.")

run_v6_contract_tests()


## Controlled live evaluation — persuasion, bargaining, then negotiation regression

V5 negotiation already has a strong positive 55-game live batch, so V6 defaults to persuasion, where seller performance was historically weakest and V6 fixes an observation-model error. Run 25–40 games at concurrency one, then test bargaining against V4's role-stratified reference. Test negotiation last as a regression check and retain V5 if V6 does not match its zero-fallback performance.

Never run two clients with the same API key. The SDK drains in-flight games before returning.


In [ ]:
from glee_sdk import GleeClient

EVALUATION_FAMILY = "persuasion"  # then "bargaining", then "negotiation"
CONCURRENCY = 1
MAX_GAMES = 30
MAX_TIME = 3600

client = GleeClient(api_key=os.environ["GLEE_API_KEY"])
before = client.stats()
print("Before:", before)
client.run(
    strategy,
    game_families=[EVALUATION_FAMILY],
    concurrency=CONCURRENCY,
    max_games=MAX_GAMES,
    max_time=MAX_TIME,
)
after = client.stats()
print("After:", after)


## Inspect decisions, fallbacks, and model state


In [ ]:
fallbacks = [entry for entry in DECISION_LOG if entry["error"]]
print("Fallback count:", len(fallbacks))
print("Recent decisions:")
display(list(DECISION_LOG)[-20:])

print("Named/game-local bargaining profiles:", len(BARGAINING_MEMORY))
print("Named/game-local negotiation profiles:", len(NEGOTIATION_MEMORY))
print("Role-separated persuasion profiles:", len(PERSUASION_MEMORY))
